# Agents SDK tracing and observability

## Learning goals

- Distinguish traces, spans, application logs, metrics, and evaluations
- Understand what the SDK traces automatically
- Configure workflow names, grouping, metadata, and sensitive-data capture
- Build a redacted application event view
- Add custom spans without exposing secrets or chain-of-thought


## Observability answers different questions

- **Trace:** what happened during one end-to-end workflow?
- **Span:** how long did one operation take, and how is it related to other operations?
- **Application log:** what product/domain event occurred under which request or user-safe identifier?
- **Metric:** how often, how slowly, or how expensively does something happen over time?
- **Evaluation:** was the output actually good or correct for the task?

A green trace means execution completed, not that the answer was correct. Observability and evaluation complement each other.


## Mental model

```mermaid
flowchart TD
  A[User request] --> B[Workflow trace]
  B --> C[Agent span]
  C --> D[Generation span]
  C --> E[Function-tool span]
  C --> F[Handoff span]
  C --> G[Guardrail span]
  B --> H[Custom application span]
  B --> I[RunResult and usage]
  I --> J[Redacted application logs]
  I --> K[Metrics and alerts]
  I --> L[Quality evaluation]
```

The SDK automatically creates runner, agent, generation, function-tool, handoff, and guardrail spans. Add custom spans only for meaningful application operations the SDK cannot infer.


## Tracing defaults and privacy

SDK tracing is enabled by default for runner calls. Generation and function-tool spans may capture their inputs and outputs. In the current SDK, sensitive-data capture is enabled by default, so production code handling private data should make an explicit decision rather than inherit that default accidentally.

Common controls:

- `RunConfig(tracing_disabled=True)` disables one run;
- `RunConfig(trace_include_sensitive_data=False)` keeps tracing while omitting sensitive generation/tool payloads;
- `OPENAI_AGENTS_DISABLE_TRACING=1` or `set_tracing_disabled(True)` disables tracing globally;
- tracing is unavailable for organizations operating under OpenAI Zero Data Retention policy.


## Configure a run explicitly

Use a stable workflow name for aggregation, a non-sensitive group ID to connect turns from one conversation, and low-cardinality metadata. Do not use an email address, API key, full prompt, or raw customer record as metadata.


In [ ]:
from agents import RunConfig

safe_run_config = RunConfig(
    workflow_name="Travel planning",
    group_id="demo-thread-001",
    trace_metadata={
        "environment": "course",
        "feature": "itinerary",
    },
    trace_include_sensitive_data=False,
    tracing_disabled=True,  # Keep offline notebook validation local.
)

print("Workflow:", safe_run_config.workflow_name)
print("Group:", safe_run_config.group_id)
print("Sensitive payload capture:", safe_run_config.trace_include_sensitive_data)


## Group several operations under one trace

Wrap multiple related runner calls or application operations in `with trace(...)`. The context manager starts and finishes the trace correctly. Custom spans nest under the active trace through context-local state and work across async concurrency.


In [ ]:
from agents import custom_span, trace


with trace(
    workflow_name="Offline travel workflow",
    group_id="demo-thread-001",
    metadata={"environment": "course"},
    disabled=True,
):
    with custom_span(
        name="validate_destination",
        data={"destination_supported": True},
        disabled=True,
    ):
        destination = "Jaipur"

print("Created disabled offline trace around:", destination)


## Application logs need an allowlist

SDK traces explain runtime mechanics. Application logs should add product meaning—request accepted, route selected, handoff completed, tool status, final outcome—using correlation IDs and redacted fields. Avoid dumping SDK objects because new fields can silently expand what gets logged.


In [ ]:
import json
from typing import Any


SENSITIVE_KEYS = {"api_key", "authorization", "prompt", "tool_arguments", "raw_output"}


def redact(value: Any) -> Any:
    if isinstance(value, dict):
        return {
            key: "[REDACTED]" if key.lower() in SENSITIVE_KEYS else redact(item)
            for key, item in value.items()
        }
    if isinstance(value, list):
        return [redact(item) for item in value]
    return value


def safe_event(
    event: str,
    *,
    request_id: str,
    details: dict[str, Any],
) -> str:
    record = {
        "event": event,
        "request_id": request_id,
        "details": redact(details),
    }
    return json.dumps(record, sort_keys=True)


print(safe_event(
    "tool_completed",
    request_id="req-demo-001",
    details={
        "tool": "get_weather",
        "status": "ok",
        "latency_ms": 42,
        "tool_arguments": {"city": "Jaipur"},
    },
))


## Derive safe events from `RunResult`

`result.new_items` is the richest SDK view for messages, tool calls/results, handoffs, and approvals. Build an allowlisted mapper that emits only item type, agent name, tool name when safe, and status. Do not serialize `raw_item`, arguments, or output by default.

Usage is available through `result.context_wrapper.usage`. Useful metrics include model requests, input/output tokens, tool counts, handoffs, guardrail tripwires, latency, retries, max-turn failures, and user-visible success. Cost and quality need separate dashboards.


In [ ]:
def safe_item_summary(item: Any) -> dict[str, str | None]:
    agent = getattr(item, "agent", None)
    return {
        "item_type": type(item).__name__,
        "agent": getattr(agent, "name", None),
    }


class FakeAgent:
    name = "Budget specialist"


class FakeHandoffItem:
    agent = FakeAgent()


print(safe_item_summary(FakeHandoffItem()))


## Optional traced live run

The live example keeps sensitive trace data disabled. For a multi-run workflow, an outer `trace()` groups the calls; individual `Runner.run` calls automatically create their nested spans.


In [ ]:
import os

from agents import Agent, ModelSettings, Runner

observed_agent = Agent(
    name="Observed travel helper",
    instructions="Answer concise travel-planning questions and state assumptions.",
    model=os.getenv("OPENAI_MODEL", "gpt-4.1-mini"),
    model_settings=ModelSettings(max_tokens=250, store=False),
)
RUN_LIVE_CALL = False

if RUN_LIVE_CALL:
    live_config = RunConfig(
        workflow_name="Travel planning",
        group_id="demo-thread-001",
        trace_metadata={"environment": "course"},
        trace_include_sensitive_data=False,
    )
    result = await Runner.run(
        observed_agent,
        "Name two architectural highlights in Jaipur.",
        max_turns=3,
        run_config=live_config,
    )
    print(result.final_output)
    print([safe_item_summary(item) for item in result.new_items])
else:
    print("Live traced run skipped.")


## Operational notes

- The default trace processor exports in batches; traces may not appear instantly.
- Long-running background workers may call `flush_traces()` after a completed outer trace when immediate export matters.
- Custom processors can add or replace exporters; they need backpressure and failure policy.
- Sampling reduces volume but must preserve enough failures and rare paths to debug.
- Retention, access control, and deletion rules apply to telemetry too.
- Do not log private chain-of-thought. Capture tool/handoff/guardrail outcomes and user-visible results.


## Exercise

1. Define five metrics for a handoff workflow and explain what each reveals.
2. Extend `redact` to handle nested authorization headers case-insensitively.
3. Design a safe `new_items` mapper for function tools and handoffs.
4. Choose a low-cardinality trace metadata schema for your domain.
5. Write an evaluation that detects bad answers even when traces show no runtime error.


## Official references

- [Agents SDK tracing](https://openai.github.io/openai-agents-python/tracing/)
- [Run results](https://openai.github.io/openai-agents-python/results/)
- [OpenAI Traces dashboard](https://platform.openai.com/traces)


## Summary

The SDK traces runner, agent, generation, tool, handoff, and guardrail operations automatically. Configure workflow/group identity and sensitive-data capture explicitly, add custom spans only for meaningful domain work, derive allowlisted application events from results, and pair runtime observability with quality evaluations.
